In [2]:
import os
import tempfile
import PIL.Image
import torch
from transformers import AutoModel, AutoTokenizer


import PIL.Image
import os
import tempfile
import torch

from transformers import AutoImageProcessor
from transformers import AutoModel, AutoTokenizer
from transformers.models.detr import DetrForSegmentation
from doclayout_yolo import YOLOv10

from src.extract_data import ExtractData

In [1]:
import cv2
from doclayout_yolo import YOLOv10

# Load the pre-trained model
model = YOLOv10("models/doclayout_yolo_docstructbench_imgsz1024.pt")

# Perform prediction
det_res = model.predict(
    "/home/proven/vlm_data_extraction/data/image/Cansu-Gökhan-Şubat 2025.jpeg",   # Image to predict
    imgsz=1024,        # Prediction image size
    conf=0.1,          # Confidence threshold
    device="cuda:0"    # Device to use (e.g., 'cuda:0' or 'cpu')
)

# Annotate and save the result
annotated_frame = det_res[0].plot(pil=True, line_width=5, font_size=20)
cv2.imwrite("result.jpg", annotated_frame)


image 1/1 /home/proven/vlm_data_extraction/data/image/Cansu-Gökhan-Şubat 2025.jpeg: 1024x768 1 title, 4 plain texts, 5 abandons, 2 figures, 3 tables, 1 table_footnote, 41.7ms
Speed: 3.8ms preprocess, 41.7ms inference, 47.3ms postprocess per image at shape (1, 3, 1024, 768)


True

In [3]:
from src.extract_data import ExtractData
from doclayout_yolo import YOLOv10


class ExtractDataFromImageYOLO(ExtractData):
    def __init__(self):
        super().__init__()
        self.model = YOLOv10(
            "models/doclayout_yolo_docstructbench_imgsz1024.pt")
        self.device = "cuda:0"
        self.conf = 0.1
        self.imgsize = 1024

        self.tokenizer = AutoTokenizer.from_pretrained(
            'ucaslcl/GOT-OCR2_0', trust_remote_code=True)
        self.ocr_model = AutoModel.from_pretrained('ucaslcl/GOT-OCR2_0', trust_remote_code=True, low_cpu_mem_usage=True,
                                                   device_map='cuda', use_safetensors=True, pad_token_id=self.tokenizer.eos_token_id)
        self.ocr_model = self.ocr_model.eval().cuda()

    def _get_yolo_attr(self, boxes):
        bboxes = boxes.xyxy.cpu().numpy()
        classes = boxes.cls.cpu().numpy()
        class_names = self.model.names if hasattr(
            self.model, 'names') else None

        _bboxes = []
        _labels = []
        for box in range(len(bboxes)):
            x1, y1, x2, y2 = bboxes[box]
            class_idx = int(classes[box])
            label = class_names[class_idx] if class_names else f"Class {class_idx}"

            _bboxes.append([int(x1), int(y1), int(x2), int(y2)])
            _labels.append(label)

        return _bboxes, _labels

    def _get_bbox_with_yolo(self, image_path):
        results = self.model.predict(
            image_path,
            imgsz= self.imgsize,
            conf=self.conf,
            device= self.device
        )
        boxes = results[0].boxes
        bboxes, labels = self._get_yolo_attr(boxes)
        return bboxes, labels

    def _crop_yolo_image(self, image_path: str, bboxes, labels) -> list:
        cropped_images = []
        image = PIL.Image.open(image_path)
        for bbox, label in zip(bboxes, labels):
            if label not in []:
                xmin, ymin, xmax, ymax = bbox
                cropped_image = image.crop((xmin, ymin, xmax, ymax))
                cropped_images.append(cropped_image)
        return cropped_images

    def extract_text_with_OCR(self, image_path: str):
        total_string = ""
        bboxes, labels = self._get_bbox_with_yolo(image_path)
        cropped_images = self._crop_yolo_image(image_path, bboxes, labels)

        with tempfile.TemporaryDirectory() as temp_dir:
            # Save each image temporarily
            for i, piece in enumerate(cropped_images):
                temp_path = os.path.join(temp_dir, f"temp_image_{i}.png")
                piece.save(temp_path)
                res = self.ocr_model.chat(
                    self.tokenizer, temp_path, ocr_type='ocr')

                total_string += f"Text Cluster_{i}: {res}" + "\n\n"
        return total_string

In [ ]:
lelelel = ExtractDataFromImageYOLO()
img= "/home/proven/vlm_data_extraction/data/image/Cansu-Gökhan-Şubat 2025.jpeg"

def extract_image(object: ExtractDataFromImageYOLO, image_file):
        text = object.extract_text_with_OCR(image_file)
        information = object.get_information(text)

        return information, text 

inf, text= extract_image(lelelel, img)



image 1/1 /home/proven/vlm_data_extraction/data/image/fatura_rıdvan_abi.jpeg: 928x1024 4 titles, 8 plain texts, 5 abandons, 1 figure, 5 tables, 4 table_captions, 14.1ms
Speed: 2.3ms preprocess, 14.1ms inference, 0.4ms postprocess per image at shape (1, 3, 928, 1024)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end gene

In [11]:
print(inf)

{
  "totalAmount": "1649.00 TL",
  "invoiceNumber": "AAA2025002559812",
  "sellerRegistrationNumber": "0080153737"
}


In [12]:
print(text)

Text Cluster_0: Fatura Adres i Rid van Altu nel Hace tte pe Universit es i Tek no kent 6. ARGE Bin as i A Blok Kat: 10 No: 29 Universit el erma halle si 06800 CAN KAYA Ankara TR Tel: 5546627369 Ver giD aires i:  V KN/ T CK N: 1111111111

Text Cluster_1: adidas

Text Cluster_2: Te slim atA d res i Rid van Altu nel Hace tte pe Universit es i Tek no kent 6. ARGE Bin asIA Blok Kat: 10 No: 29 Universit el erma hall es i 06800 CAN KAYA Ankara TR Tel: 5546627369

Text Cluster_3: 1/1

Text Cluster_4: Sira DurNodu Beden DurnAdi Adet Birim KDV Birim Toplam No Fiyat Orani Fiyat Tutar (Net) (KDV'I) 1 A4848 S AdicolorClassics3-StripesTisrt 1 1.499,09 10,00% 1.649,00 1.649,00 Indirim -149,91 -164,90 -164,90

Text Cluster_5: ladeedece@inizurinleriYurticiKargoile: OmsanLojistikA.S.TuziaDepo-Sifamah.SekerpınarCad.No:25Tuzia/STANBULadresinegonderebilirsiniz.Urunleri,ladesebebinin isaretendi@ifaturailebirlikeponderenizsirechizandmarkacisindan6nemlidir

Text Cluster_6: e- Ars iv Fatu ra

Text Cluster_7: l